kernel: mlenv

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.ndimage import zoom
from functions_for_goes import crop_goes_to_latlon_box, convert_radiance_to_brightness_temp

# Run JUDGE on example GOES scene

In [ ]:
import timm
from torch import nn
import torch
from torch.utils.data import DataLoader
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functions_for_goes import crop_goes_to_latlon_box, convert_radiance_to_brightness_temp
from judge_goes import judge_GOES, setup_region_parameters


time = '1800'
day='26'
month='07'
year='2025'

region='SEP'
data_dir=f'/home/jmayer/Object_Data/GOES19/{year}/{month}/'
model_path='/home/jmayer/ACI_research/Code/mesoscale_structure/step2_judge_for_goes/finetuned_goes_model.pth'
device_num='1'
plot_path='./../analysis/figures/'


# Setup region parameters
lat_bounds, lon_bounds, times = setup_region_parameters(region)


df_results = pd.DataFrame(columns=['datetime', 'y_idx', 'x_idx', 'lon_mid', 'lat_mid', 'ice_flag', 'cats', 'cert'])

# Process each day and time
date = f'{year}{month}{day}'
ts = pd.to_datetime(date + time)
    
print('.' * 85)
print(f"Processing date {date} time {time}...")

# Load and crop GOES data
print("Loading C11 data...")
with xr.open_dataset(f'{data_dir}/goes19_C11_RadF_{date}_{time}.nc') as ds:
    ds = convert_radiance_to_brightness_temp(ds)
    ds_11_cropped = crop_goes_to_latlon_box(ds, lat_bounds, lon_bounds)

print("Loading C14 data...")
with xr.open_dataset(f'{data_dir}/goes19_C14_RadF_{date}_{time}.nc') as ds:
    ds = convert_radiance_to_brightness_temp(ds)
    ds_14_cropped = crop_goes_to_latlon_box(ds, lat_bounds, lon_bounds)

print("Loading C15 data...")
with xr.open_dataset(f'{data_dir}/goes19_C15_RadF_{date}_{time}.nc') as ds:
    ds = convert_radiance_to_brightness_temp(ds)
    ds_15_cropped = crop_goes_to_latlon_box(ds, lat_bounds, lon_bounds)

print("Loading COD data...")
with xr.open_dataset(f'{data_dir}/goes19_COD2KMF_{date}_{time}.nc') as ds:
    ds_COD_cropped = crop_goes_to_latlon_box(ds, lat_bounds, lon_bounds)

print('.' * 85)
print('Running JUDGE...')
cats, cert, scenes_params = judge_GOES(
    ds_11_cropped['BT'], ds_14_cropped['BT'], ds_15_cropped['BT'], 
    ds_COD_cropped['COD'], model_path, device_num
)

# Save results to dataframe
df = pd.DataFrame(scenes_params)
df['cats'] = cats
df['cert'] = cert
df['datetime'] = pd.to_datetime(date + time)
df_results = pd.concat([df_results, df], ignore_index=True)

/home/jmayer/miniconda3/envs/mlenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


.....................................................................................
Processing date 20250726 time 1800...
Loading C11 data...


getfattr: /home/jmayer/Object_Data/GOES19/2025/07/goes19_C11_RadF_20250726_1800.nc: Operation not supported
/home/jmayer/ACI_research/Code/mesoscale_structure/step2_judge_for_goes/functions_for_goes.py:112: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  y_max = min(ds.dims['y'] - 1, y_max + padding)
/home/jmayer/ACI_research/Code/mesoscale_structure/step2_judge_for_goes/functions_for_goes.py:114: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  x_max = min(ds.dims['x'] - 1, x_max + padding)


Original shape: FrozenMappingWarningOnValuesAccess({'y': 5424, 'x': 5424, 'number_of_time_bounds': 2, 'number_of_image_bounds': 2, 'band': 1, 'number_of_harmonization_coefficients': 3, 'num_star_looks': 24})
Cropped shape: FrozenMappingWarningOnValuesAccess({'y': 1197, 'x': 1094, 'number_of_time_bounds': 2, 'number_of_image_bounds': 2, 'band': 1, 'number_of_harmonization_coefficients': 3, 'num_star_looks': 24})
Lat range: -25.56 to -2.29
Lon range: -97.96 to -74.81
Loading C14 data...


getfattr: /home/jmayer/Object_Data/GOES19/2025/07/goes19_C14_RadF_20250726_1800.nc: Operation not supported
/home/jmayer/ACI_research/Code/mesoscale_structure/step2_judge_for_goes/functions_for_goes.py:112: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  y_max = min(ds.dims['y'] - 1, y_max + padding)
/home/jmayer/ACI_research/Code/mesoscale_structure/step2_judge_for_goes/functions_for_goes.py:114: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  x_max = min(ds.dims['x'] - 1, x_max + padding)


Original shape: FrozenMappingWarningOnValuesAccess({'y': 5424, 'x': 5424, 'number_of_time_bounds': 2, 'number_of_image_bounds': 2, 'band': 1, 'number_of_harmonization_coefficients': 3, 'num_star_looks': 24})
Cropped shape: FrozenMappingWarningOnValuesAccess({'y': 1197, 'x': 1094, 'number_of_time_bounds': 2, 'number_of_image_bounds': 2, 'band': 1, 'number_of_harmonization_coefficients': 3, 'num_star_looks': 24})
Lat range: -25.56 to -2.29
Lon range: -97.96 to -74.81
Loading C15 data...


getfattr: /home/jmayer/Object_Data/GOES19/2025/07/goes19_C15_RadF_20250726_1800.nc: Operation not supported
/home/jmayer/ACI_research/Code/mesoscale_structure/step2_judge_for_goes/functions_for_goes.py:112: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  y_max = min(ds.dims['y'] - 1, y_max + padding)
/home/jmayer/ACI_research/Code/mesoscale_structure/step2_judge_for_goes/functions_for_goes.py:114: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  x_max = min(ds.dims['x'] - 1, x_max + padding)


Original shape: FrozenMappingWarningOnValuesAccess({'y': 5424, 'x': 5424, 'number_of_time_bounds': 2, 'number_of_image_bounds': 2, 'band': 1, 'number_of_harmonization_coefficients': 3, 'num_star_looks': 24})
Cropped shape: FrozenMappingWarningOnValuesAccess({'y': 1197, 'x': 1094, 'number_of_time_bounds': 2, 'number_of_image_bounds': 2, 'band': 1, 'number_of_harmonization_coefficients': 3, 'num_star_looks': 24})
Lat range: -25.56 to -2.29
Lon range: -97.96 to -74.81
Loading COD data...


getfattr: /home/jmayer/Object_Data/GOES19/2025/07/goes19_COD2KMF_20250726_1800.nc: Operation not supported


KeyboardInterrupt: 

In [4]:
with xr.open_dataset(f'{data_dir}/goes19_C11_RadF_{date}_{time}.nc') as ds:
    ds.load()

getfattr: /home/jmayer/Object_Data/GOES19/2025/07/goes19_C11_RadF_20250726_1800.nc: Operation not supported


In [5]:
convert_radiance_to_brightness_temp(ds)

<xarray.Dataset> Size: 353MB
Dimensions:                                           (y: 5424, x: 5424,
                                                       number_of_time_bounds: 2,
                                                       number_of_image_bounds: 2,
                                                       band: 1,
                                                       number_of_harmonization_coefficients: 3,
                                                       num_star_looks: 24)
Coordinates:
    t                                                 datetime64[ns] 8B 2025-...
  * y                                                 (y) float32 22kB 0.1518...
  * x                                                 (x) float32 22kB -0.151...
    y_image                                           float32 4B 0.0
    x_image                                           float32 4B 0.0
    band_id                                           (band) int8 1B 11
    band_wavelength                                   (band) float32 4B 8.44
    t_star_look                                       (num_star_looks) datetime64[ns] 192B ...
    band_wavelength_star_look                         (num_star_looks) float32 96B ...
Dimensions without coordinates: number_of_time_bounds, number_of_image_bounds,
                                band, number_of_harmonization_coefficients,
                                num_star_looks
Data variables: (12/40)
    Rad                                               (y, x) float32 118MB na...
    DQF                                               (y, x) float32 118MB na...
    time_bounds                                       (number_of_time_bounds) datetime64[ns] 16B ...
    goes_imager_projection                            int32 4B -2147483647
    y_image_bounds                                    (number_of_image_bounds) float32 8B ...
    x_image_bounds                                    (number_of_image_bounds) float32 8B ...
    ...                                                ...
    processing_parm_version_container                 int32 4B -2147483647
    algorithm_product_version_container               int32 4B -2147483647
    star_id                                           (num_star_looks) float32 96B ...
    channel_integration_time                          float64 8B nan
    channel_gain_field                                float64 8B nan
    BT                                                (y, x) float32 118MB na...
Attributes: (12/30)
    naming_authority:          gov.nesdis.noaa
    Conventions:               CF-1.7
    standard_name_vocabulary:  CF Standard Name Table (v35, 20 July 2016)
    institution:               DOC/NOAA/NESDIS > U.S. Department of Commerce,...
    project:                   GOES
    production_site:           WCDAS
    ...                        ...
    timeline_id:               ABI Mode 6
    date_created:              2025-07-26T18:09:57.1Z
    time_coverage_start:       2025-07-26T18:00:20.6Z
    time_coverage_end:         2025-07-26T18:09:51.5Z
    LUT_Filenames:             SpaceLookParams(FM4A_CDRL79RevA_DO_14_00_00)-7...
    id:                        23db7c14-8ddf-4cf0-af68-8cef338ea0a0

In [6]:
ds

<xarray.Dataset> Size: 353MB
Dimensions:                                           (y: 5424, x: 5424,
                                                       number_of_time_bounds: 2,
                                                       number_of_image_bounds: 2,
                                                       band: 1,
                                                       number_of_harmonization_coefficients: 3,
                                                       num_star_looks: 24)
Coordinates:
    t                                                 datetime64[ns] 8B 2025-...
  * y                                                 (y) float32 22kB 0.1518...
  * x                                                 (x) float32 22kB -0.151...
    y_image                                           float32 4B 0.0
    x_image                                           float32 4B 0.0
    band_id                                           (band) int8 1B 11
    band_wavelength                                   (band) float32 4B 8.44
    t_star_look                                       (num_star_looks) datetime64[ns] 192B ...
    band_wavelength_star_look                         (num_star_looks) float32 96B ...
Dimensions without coordinates: number_of_time_bounds, number_of_image_bounds,
                                band, number_of_harmonization_coefficients,
                                num_star_looks
Data variables: (12/40)
    Rad                                               (y, x) float32 118MB na...
    DQF                                               (y, x) float32 118MB na...
    time_bounds                                       (number_of_time_bounds) datetime64[ns] 16B ...
    goes_imager_projection                            int32 4B -2147483647
    y_image_bounds                                    (number_of_image_bounds) float32 8B ...
    x_image_bounds                                    (number_of_image_bounds) float32 8B ...
    ...                                                ...
    processing_parm_version_container                 int32 4B -2147483647
    algorithm_product_version_container               int32 4B -2147483647
    star_id                                           (num_star_looks) float32 96B ...
    channel_integration_time                          float64 8B nan
    channel_gain_field                                float64 8B nan
    BT                                                (y, x) float32 118MB na...
Attributes: (12/30)
    naming_authority:          gov.nesdis.noaa
    Conventions:               CF-1.7
    standard_name_vocabulary:  CF Standard Name Table (v35, 20 July 2016)
    institution:               DOC/NOAA/NESDIS > U.S. Department of Commerce,...
    project:                   GOES
    production_site:           WCDAS
    ...                        ...
    timeline_id:               ABI Mode 6
    date_created:              2025-07-26T18:09:57.1Z
    time_coverage_start:       2025-07-26T18:00:20.6Z
    time_coverage_end:         2025-07-26T18:09:51.5Z
    LUT_Filenames:             SpaceLookParams(FM4A_CDRL79RevA_DO_14_00_00)-7...
    id:                        23db7c14-8ddf-4cf0-af68-8cef338ea0a0

In [3]:
df_results

,datetime,y_idx,x_idx,lon_mid,lat_mid,ice_flag,cats,cert,cloud_cover,ot_gt_1,ot_gt_5
0,2025-07-26 18:00:00,64,64,-94.553077,-2.903803,0.0,3,0.612850,0.998535,0.997070,0.727783
1,2025-07-26 18:00:00,64,128,-93.921278,-2.901942,0.0,3,0.835426,0.990479,0.985840,0.728027
2,2025-07-26 18:00:00,64,192,-93.293226,-2.900151,0.0,3,0.740824,0.986816,0.980713,0.762207
3,2025-07-26 18:00:00,64,256,-92.668755,-2.898428,0.0,3,0.710848,0.984375,0.976807,0.795654
4,2025-07-26 18:00:00,64,320,-92.047694,-2.896772,0.0,3,0.804815,0.984619,0.978027,0.806152
...,...,...,...,...,...,...,...,...,...,...,...
1183,2025-07-26 18:00:00,2304,1856,-78.136248,-24.277994,0.0,3,0.657959,0.862305,0.807129,0.484375
1184,2025-07-26 18:00:00,2304,1920,-77.492139,-24.275638,0.0,3,0.564197,0.913818,0.862061,0.657959
1185,2025-07-26 18:00:00,2304,1984,-76.848549,-24.273823,0.0,3,0.896639,0.915771,0.852295,0.646240
1186,2025-07-26 18:00:00,2304,2048,-76.205346,-24.272546,0.0,3,0.592936,0.932129,0.864258,0.563721


# Plot 

In [4]:
pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
def create_ocean_mask_optimized(df0, lat_col='latitude', lon_col='longitude'):
    import numpy as np
    import geopandas as gpd

    lats = df0[lat_col].values
    lons = df0[lon_col].values

    # Load Natural Earth land polygons directly from URL (geopandas 1.0+ compatible)
    land = gpd.read_file('https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_land.zip')
    land_union = land.geometry.union_all()

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(lons, lats),
        crs='EPSG:4326'
    )
    is_over_land = points.geometry.within(land_union).values

    ocean_mask = ~is_over_land
    print(f"Final result: {np.sum(ocean_mask)} total ocean points out of {len(df0)}")
    return ocean_mask

df_results['ocean_mask'] = create_ocean_mask_optimized(df_results, lat_col='lat_mid', lon_col='lon_mid')

Final result: 1057 total ocean points out of 1188


In [6]:
df_results

,datetime,y_idx,x_idx,lon_mid,lat_mid,ice_flag,cats,cert,cloud_cover,ot_gt_1,ot_gt_5,ocean_mask
0,2025-07-26 18:00:00,64,64,-94.553077,-2.903803,0.0,3,0.612850,0.998535,0.997070,0.727783,True
1,2025-07-26 18:00:00,64,128,-93.921278,-2.901942,0.0,3,0.835426,0.990479,0.985840,0.728027,True
2,2025-07-26 18:00:00,64,192,-93.293226,-2.900151,0.0,3,0.740824,0.986816,0.980713,0.762207,True
3,2025-07-26 18:00:00,64,256,-92.668755,-2.898428,0.0,3,0.710848,0.984375,0.976807,0.795654,True
4,2025-07-26 18:00:00,64,320,-92.047694,-2.896772,0.0,3,0.804815,0.984619,0.978027,0.806152,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1183,2025-07-26 18:00:00,2304,1856,-78.136248,-24.277994,0.0,3,0.657959,0.862305,0.807129,0.484375,True
1184,2025-07-26 18:00:00,2304,1920,-77.492139,-24.275638,0.0,3,0.564197,0.913818,0.862061,0.657959,True
1185,2025-07-26 18:00:00,2304,1984,-76.848549,-24.273823,0.0,3,0.896639,0.915771,0.852295,0.646240,True
1186,2025-07-26 18:00:00,2304,2048,-76.205346,-24.272546,0.0,3,0.592936,0.932129,0.864258,0.563721,True


In [7]:
df_results = df_results[df_results.ocean_mask==1]

In [8]:
# Visualize results
#palette_goes_cats = {0: 'purple', 1: 'steelblue', 2: 'green', 3: 'orange', 4: 'red', 5: 'hotpink', np.nan: 'grey'}
palette_goes_cats = ['#2ca02c', 'blue', 'orange', '#d62728', '#e377c2', '#8c564b', 'white']
fig, axs = plt.subplots(1, 2, figsize=(21, 10))
COD_interpolated = zoom(ds_COD_cropped['COD'].values, 2, order=1)
cot_plot = axs[0].imshow(COD_interpolated, vmin=0, vmax=25, cmap='Greys_r')
BT_14_interpolated = zoom(ds_14_cropped['BT'].values, 2, order=1)
axs[1].imshow(BT_14_interpolated, vmax=290, vmin=280, cmap='Greys')
for ax in axs:
    sns.scatterplot(ax=ax, x=df_results['x_idx'], y=df_results['y_idx'], 
                    hue=df_results['cats'], size=12, lw=0, palette=palette_goes_cats)
    
    # add rectangle showing the size of the 128 x 128 km scenes
    import matplotlib.patches as mpatches
    i = 60
    x_mid,y_mid = (df_results['x_idx'].iloc[i], df_results['y_idx'].iloc[i])
    rect1 = mpatches.Rectangle((x_mid - 64, y_mid - 64), (128), (128), linewidth=3, edgecolor='yellow', facecolor='none', zorder=2)
    ax.add_patch(rect1)

axs[0].axis('off')
axs[1].axis('off')
cbar0 = plt.colorbar(cot_plot, ax=axs[0], fraction=0.03, pad=0.01, orientation='horizontal')
cbar0.set_label('GOES cloud optical thickness', fontsize=12)
axs[0].set_title(f'{year}-{month}-{day} {time[0:2]} UTC', fontsize=16)

cats_names = ['Solid Stratus', 'Closed MCC', 'Open MCC', 'Disorg. MCC', 'Clustered Cu.', 'Suppr. Cu.', 'Ice cloud']
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', label=cats_names[i], 
                        markerfacecolor=palette_goes_cats[i], markersize=8, markeredgecolor='white', markeredgewidth=0.5) 
                    for i in range(6)]
axs[0].legend(handles=legend_handles, title='CNN Category', bbox_to_anchor=(0.99, 0.99), loc='upper right', fontsize=10)

plt.savefig(os.path.join(plot_path, f'example_judge_goes.png'))
plt.close()

/tmp/ipykernel_1481187/1636785047.py:10: UserWarning: The palette list has more values (7) than needed (6), which may not be intended.
  sns.scatterplot(ax=ax, x=df_results['x_idx'], y=df_results['y_idx'],
/tmp/ipykernel_1481187/1636785047.py:10: UserWarning: The palette list has more values (7) than needed (6), which may not be intended.
  sns.scatterplot(ax=ax, x=df_results['x_idx'], y=df_results['y_idx'],
